# CHIMERA-TTS 🔥 — T4 Prototype
Runtime consigliato: **Colab T4 GPU free**. Questa run allena il prototype 85M (stesso codice del 3B) e genera i primi audio.

In [ ]:
!nvidia-smi -L

In [ ]:
!test -d /content/AI-Model-TTS && (cd /content/AI-Model-TTS && git pull) || git clone https://github.com/Taiger7196/AI-Model-TTS.git /content/AI-Model-TTS
%cd /content/AI-Model-TTS
!git checkout arena/01a08c88-ai-model-tts
!pip install -q -r requirements.txt

In [ ]:
!python scripts/count_params.py

In [ ]:
# Smoke test: 4 step su dati sintetici (no download, ~1 min anche su CPU)
!python - <<'EOF'
import sys; sys.path.insert(0, "src")
import torch
from torch.utils.data import DataLoader
from chimera import ChimeraConfig, ChimeraTTS
from chimera.data import TinyDemoDataset, collate
from chimera.trainer import AcousticTrainer, seed_all
seed_all(0)
cfg = ChimeraConfig(max_steps=4, batch_size=2, grad_accum=2, log_every=1,
                    ckpt_every=100, sample_every=100, out_dir="/tmp/chim_smoke")
loader = DataLoader(TinyDemoDataset(cfg, n=16), batch_size=2, collate_fn=collate)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)
AcousticTrainer(cfg, ChimeraTTS(cfg), loader, loader, dev).train()
print("SMOKE OK")
EOF

In [ ]:
# Training reale su LibriTTS dev-clean (subset 2000 utterance, ~2-3h su T4).
# Aumenta --max-steps per la run di convergenza. Se Colab si disconnette,
# rilancia con --resume checkpoints/chimera-tiny/step_XXXXXX.pt
!python scripts/train.py --config configs/tiny_100m.yaml --max-items 2000 --max-steps 2000

In [ ]:
# Ascolta l'ultimo sample generato durante il training
import glob
from IPython.display import Audio
gens = sorted(glob.glob("checkpoints/chimera-tiny/samples/*_gen.wav"))
print(gens[-1] if gens else "no samples yet")
Audio(gens[-1]) if gens else None

## Next
- Se i sample migliorano → alza `--max-steps` e full dev-clean, poi train-clean-100
- Poi: vocoder HiFi-GAN (fase 1) → codec + scale-up 3B (fase 2)
- Sintesi custom: `python scripts/synth.py --ckpt <ckpt> --ref voice.wav --text "..." --out out.wav`